In [1]:
from pipeline_components import ExamFormatterGemini, ExamGrader, DocumentReader

# Pipeline

## V1 - Synchronous

In [2]:
from llama_cloud import LlamaCloud
from google import genai
import os
from typing import List
from dotenv import load_dotenv
import pymupdf
import json

In [3]:
class ExamGradingPipeline:

    def __init__(self, gemini_client: genai.Client, llama_cloud: LlamaCloud):
        self.document_reader = DocumentReader(llama_cloud)
        self.exam_formatter = ExamFormatterGemini(gemini_client)
        self.exam_grader = ExamGrader(gemini_client)

    def grade_exams(
            self, 
            exam_paper: bytes | str, 
            answer_papers: List[bytes] | List[str], 
            answer_key: bytes | str,
            instructions: str
        ) -> tuple[List[str], List[str]]:
        exam_paper_text = self.document_reader.read_doc(exam_paper) if isinstance(exam_paper, bytes) else exam_paper
        formatted_exam_paper = self.exam_formatter.format_exam_paper(exam_paper_text)
        print("Formatted Exam paper: ")
        print(formatted_exam_paper)

        answer_key_text = self.document_reader.read_doc(answer_key) if isinstance(answer_key, bytes) else answer_key
        formatted_answer_key = self.exam_formatter.format_answer_key(answer_key_text, formatted_exam_paper)
        print("Formatted Answer Key: ")
        print(formatted_answer_key)

        graded_answer_papers = []
        failed_answer_papers = []
       
        for answer_paper in answer_papers:
            answer_paper_text = self.document_reader.read_doc(answer_paper) if isinstance(answer_paper, bytes) else answer_paper
            formatted_answer_paper = self.exam_formatter.format_answer_paper(answer_paper_text, formatted_exam_paper)
            print("Formatted Answer paper: ")
            print(formatted_answer_paper)
            graded_answer_paper = self.exam_grader.grade_exam_paper(formatted_exam_paper, formatted_answer_paper, formatted_answer_key, instructions)
            if self._validate_marks_range(graded_answer_paper, formatted_exam_paper):
                graded_answer_papers.append(graded_answer_paper)
            else:
                failed_answer_papers.append(graded_answer_paper)
            
        return (graded_answer_papers, failed_answer_papers)

    def _validate_marks_range(self, graded_answer_paper: str, exam_paper: str) -> bool:
        answer_paper_dict = json.loads(graded_answer_paper)
        graded_objective_questions = answer_paper_dict["objective"]
        graded_subjective_questions = answer_paper_dict["subjective"]
        exam_paper_dict = json.loads(exam_paper)
        objective_questions = exam_paper_dict["objective"]
        subjective_questions = exam_paper_dict["subjective"]

        if not self._in_range(answer_paper_dict["marks_obtained"], exam_paper_dict["marks"]):
            return False

        for i in range(len(graded_objective_questions)):
            if not self._in_range(graded_objective_questions[i]["marks_obtained"], objective_questions[i]["marks"]):
                return False

        return self._validate_nested_questions(graded_subjective_questions, subjective_questions)
            
            
    def _validate_nested_questions(self, graded_questions: List, exam_questions: List) -> bool:
        for i in range(len(graded_questions)):
            if not self._in_range(graded_questions[i]["marks_obtained"], exam_questions[i]["marks"]):
                return False

            graded_subquestions = graded_questions[i]["subquestions"]
            subquestions = exam_questions[i]["subquestions"]
            if subquestions is not None:
                if not self._validate_nested_questions(graded_questions, subquestions):
                    return False

        return True

    def _in_range(self, value: float, max: float, min: float = 0.0) -> bool:
        if value < min or value > max:
            return False
        else:
            return True


### Testing

In [4]:
load_dotenv()

True

In [5]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [6]:
llama_cloud = LlamaCloud()

In [7]:
pipe = ExamGradingPipeline(gemini_client, llama_cloud)

In [8]:
exam_paper = pymupdf.open("test_documents/exam_paper.pdf").tobytes()
answer_paper_1 = pymupdf.open("test_documents/handwritten_exam_1.pdf").tobytes()
answer_paper_2 = pymupdf.open("test_documents/handwritten_exam_2.pdf").tobytes()
answer_paper_3 = pymupdf.open("test_documents/handwritten_exam_3.pdf").tobytes()
answer_paper_4 = pymupdf.open("test_documents/handwritten_exam_4.pdf").tobytes()

In [9]:
answer_key = """
Answer key: Question_Number,Type,Correct_Answer
1,MCQ,B
2,MCQ,C
3,MCQ,A
4,MCQ,C
5,MCQ,B
6,MCQ,A
7,MCQ,A
8,MCQ,C
9,MCQ,A
10,MCQ,B
11,MCQ,B
12,MCQ,C
13,MCQ,B
14,MCQ,C
15,MCQ,B
16,MCQ,C
17,MCQ,B
18,MCQ,C
19,MCQ,B
20,MCQ,B
21,Short_Answer,"A type of ML where the model is trained on labeled data with known inputs and outputs."
22,Short_Answer,"Overfitting; the model has memorized the training data but failed to generalize."
23,Short_Answer,"A statistical process for estimating the relationships between a dependent variable and one or more independent variables."
24,Short_Answer,"Data quality/missing values, overfitting/underfitting, data collection, or time consumption."
25,Short_Answer,"Healthcare/disease prediction, Finance/fraud detection, E-commerce recommendation systems."
26,Short_Answer,"Structured is organized in rows/columns (SQL, Excel); Semi-structured has some markers but isn't in a grid (JSON, XML, HTML)."
27,Short_Answer,"Detection: Z-score, IQR, Scatter plots, Box plots. Handling: Trimming, Capping, or Imputation."
28,Short_Answer,"The steps taken to clean and transform raw data into a format suitable for a machine learning model."
29,Short_Answer,"K-NN is distance-based; features with larger scales will dominate the distance calculation if not normalized."
30,Short_Answer,"By grouping values into bins or intervals, e.g., turning ages 1-100 into 'Child,' 'Adult,' 'Senior.'"
31,Short_Answer,"The predefined number of clusters or groups the data will be divided into."
32,Short_Answer,"K in K-NN, Learning Rate, Number of Estimators/Trees, Max Depth, or Batch Size."
33,Short_Answer,"An optimization algorithm used to minimize the cost function by iteratively moving toward the steepest descent."
34,Short_Answer,"Discussion on maximizing the margin and ensuring no data points fall within the 'gutter' or margin."
35,Short_Answer,"It transforms data into a higher-dimensional space to find a non-linear decision boundary."
 
"""

In [10]:
instructions = """
Check like a mid teacher not very strict not very lenient.  
"""

In [11]:
success, failed = pipe.grade_exams(
    exam_paper,
    [answer_paper_1, answer_paper_2, answer_paper_3],
    answer_key,
    instructions
)

Formatted Exam paper: 
{"subject":"Data Science","course_id":null,"date":null,"marks":55.0,"objective":[{"question_id":"Q(1)","marks":1.0,"mcq_statement":"Which of the following is an assumption of Linear Regression?\nA. High multicollinearity\nB. Little or no autocorrelation\nC. Heteroscedasticity\nD. All of the mentioned"},{"question_id":"Q(2)","marks":1.0,"mcq_statement":"In a perfectly fitted regression model, the sum of residuals (errors) should be:\nA. Maximum\nB. Minimum\nC. Zero\nD. Unknown"},{"question_id":"Q(3)","marks":1.0,"mcq_statement":"Which distance metric is specifically used for categorical variables or strings of equal length?\nA. Hamming Distance\nB. Euclidean Distance\nC. Manhattan Distance\nD. Minkowski Distance"},{"question_id":"Q(4)","marks":1.0,"mcq_statement":"Supervised learning can be split into which two main categories?\nA. Classification\nB. Regression\nC. Both A & B\nD. None of the mentioned"},{"question_id":"Q(5)","marks":1.0,"mcq_statement":"Which of t

In [12]:
success

['{"roll_no": "", "name": null, "objective": [{"question_id": "Q(1)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(2)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(3)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(4)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(5)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(6)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(7)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(8)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(9)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(10)", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q(11)", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q(12)", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q(13)", "marks_obtained": 1.0, "reasoning": "Corre

## V2 - Asynchronous

In [13]:
import asyncio
import time
import threading

In [18]:
class ExamGradingPipelineAsync:

    def __init__(self, gemini_client: genai.Client, llama_cloud: LlamaCloud):
        self.document_reader = DocumentReader(llama_cloud)
        self.exam_formatter = ExamFormatterGemini(gemini_client)
        self.exam_grader = ExamGrader(gemini_client)

    async def grade_exams(
            self, 
            exam_paper: bytes | str, 
            answer_papers: List[bytes] | List[str], 
            answer_key: bytes | str,
            instructions: str,
            concurrency: int = 3,
        ) -> tuple[List[str], List[str]]:
        exam_paper_text = self.document_reader.read_doc(exam_paper) if isinstance(exam_paper, bytes) else exam_paper
        formatted_exam_paper = self.exam_formatter.format_exam_paper(exam_paper_text)
        print("Formatted Exam paper: ")
        print(formatted_exam_paper)

        answer_key_text = self.document_reader.read_doc(answer_key) if isinstance(answer_key, bytes) else answer_key
        formatted_answer_key = self.exam_formatter.format_answer_key(answer_key_text, formatted_exam_paper)
        print("Formatted Answer Key: ")
        print(formatted_answer_key)

        graded_answer_papers = []
        failed_answer_papers = []

        
        await self._grade_exams_concurrently(answer_papers, formatted_exam_paper, formatted_answer_key, instructions, concurrency, graded_answer_papers, failed_answer_papers)
        

        return (graded_answer_papers, failed_answer_papers)

    async def _grade_exam(
            self, 
            answer_paper: bytes | str, 
            formatted_exam_paper: str, 
            formatted_answer_key: str, 
            instructions: str, 
            graded_answer_papers: List[str], 
            failed_answer_papers: List[str]
        ):
        self.log_calls("Calling Grade exam")
        answer_paper_text = await asyncio.to_thread(self.document_reader.read_doc, answer_paper) if isinstance(answer_paper, bytes) else answer_paper
        self.log_calls("Formatting")
        formatted_answer_paper = await asyncio.to_thread(self.exam_formatter.format_answer_paper, answer_paper_text, formatted_exam_paper)
        self.log_calls("Formatted Answer paper: ")
        print(formatted_answer_paper)
        self.log_calls("Grading Answer paper")
        graded_answer_paper = await asyncio.to_thread(self.exam_grader.grade_exam_paper, formatted_exam_paper, formatted_answer_paper, formatted_answer_key, instructions)
        if self._validate_marks_range(graded_answer_paper, formatted_exam_paper):
            graded_answer_papers.append(graded_answer_paper)
        else:
            failed_answer_papers.append(graded_answer_paper)

    async def _grade_exams_concurrently(
            self, 
            answer_papers: List[bytes | str], 
            formatted_exam_paper: str, 
            formatted_answer_key: str, 
            instructions: str, 
            concurrency: int, 
            graded_answer_papers: List[str], 
            failed_answer_papers: List[str]
        ):
        # Iterate through all papers and group them into batches of size concurrency, then grade the group concurrently
        counter = 0
        while counter < len(answer_papers):
            # Create a task group to grade the concurreny number of answer papers concurrently
            async with asyncio.TaskGroup() as tg:
                for i in range(0, concurrency):
                    counter += 1
                    # Break the loop if the counter exceeds the number of answer papers
                    if counter > len(answer_papers):
                        break
                    tg.create_task(self._grade_exam(answer_papers[counter-1], formatted_exam_paper, formatted_answer_key, instructions, graded_answer_papers, failed_answer_papers))
                    
        
    def _validate_marks_range(self, graded_answer_paper: str, exam_paper: str) -> bool:
        answer_paper_dict = json.loads(graded_answer_paper)
        graded_objective_questions = answer_paper_dict["objective"]
        graded_subjective_questions = answer_paper_dict["subjective"]
        exam_paper_dict = json.loads(exam_paper)
        objective_questions = exam_paper_dict["objective"]
        subjective_questions = exam_paper_dict["subjective"]

        if not self._in_range(answer_paper_dict["marks_obtained"], exam_paper_dict["marks"]):
            return False

        for i in range(len(graded_objective_questions)):
            if not self._in_range(graded_objective_questions[i]["marks_obtained"], objective_questions[i]["marks"]):
                return False

        return self._validate_nested_questions(graded_subjective_questions, subjective_questions)
            
            
    def _validate_nested_questions(self, graded_questions: List, exam_questions: List) -> bool:
        for i in range(len(graded_questions)):
            if not self._in_range(graded_questions[i]["marks_obtained"], exam_questions[i]["marks"]):
                return False

            graded_subquestions = graded_questions[i]["subquestions"]
            subquestions = exam_questions[i]["subquestions"]
            if subquestions is not None:
                if not self._validate_nested_questions(graded_subquestions, subquestions):
                    return False

        return True

    def _in_range(self, value: float, max: float, min: float = 0.0) -> bool:
        if value < min or value > max:
            return False
        else:
            return True

    def log_calls(self, label):
        print(
            f"{time.perf_counter():.2f} | "
            f"Thread={threading.current_thread().name} | "
            f"{label}"
        )


In [19]:
async_pipe = ExamGradingPipelineAsync(gemini_client, llama_cloud)

In [ ]:
success, failed = await async_pipe.grade_exams(
    exam_paper,
    [answer_paper_1, answer_paper_2, answer_paper_3],
    answer_key,
    instructions,
    3
)

Formatted Exam paper: 
{"subject":"Data Science","course_id":null,"date":null,"marks":55.0,"objective":[{"question_id":"Q1","marks":1.0,"mcq_statement":"Which of the following is an assumption of Linear Regression?\nA. High multicollinearity\nB. Little or no autocorrelation\nC. Heteroscedasticity\nD. All of the mentioned"},{"question_id":"Q2","marks":1.0,"mcq_statement":"In a perfectly fitted regression model, the sum of residuals (errors) should be:\nA. Maximum\nB. Minimum\nC. Zero\nD. Unknown"},{"question_id":"Q3","marks":1.0,"mcq_statement":"Which distance metric is specifically used for categorical variables or strings of equal length?\nA. Hamming Distance\nB. Euclidean Distance\nC. Manhattan Distance\nD. Minkowski Distance"},{"question_id":"Q4","marks":1.0,"mcq_statement":"Supervised learning can be split into which two main categories?\nA. Classification\nB. Regression\nC. Both A & B\nD. None of the mentioned"},{"question_id":"Q5","marks":1.0,"mcq_statement":"Which of the followi

In [21]:
success

['{"roll_no": "", "name": null, "objective": [{"question_id": "Q1", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q2", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q3", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q4", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q5", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q6", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q7", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q8", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q9", "marks_obtained": 0.0, "reasoning": "Incorrect"}, {"question_id": "Q10", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q11", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q12", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q13", "marks_obtained": 1.0, "reasoning": "Correct"}, {"question_id": "Q14", "marks_ob